# DNA Methylation Stacked Ensemble Inference Pipeline (Portable)

This notebook implements a portable prediction pipeline for evaluating new DNA methylation datasets on the pre-trained FHS60 stacked ensemble clock. 

It loads all models relatively from the `pretrained_models/` directory in this repository:
1. **CpG feature selection**: 190 CpG features selected by CoxNet.
2. **Standardization scaler**: Centers and scales the CpG methylation inputs.
3. **Five Base Learners**: Random Survival Forest (RSF), Gradient Boosting Machine (GBM), Fast Kernel SVM (SVM), Weibull AFT, and MTLR.
4. **Stacking Meta-Learner**: A DeepSurv-style MLP trained on Out-of-Fold (OOF) base risk predictions.
5. **Age Regressor**: Calibrates the final epigenetic age prediction and computes age acceleration.


### 1. Imports and Pickle Compatibility Patches


In [ ]:
import sys
import sklearn.base
import numpy as np

# Pickle compatibility patch for sksurv version mismatch
class DummySurvivalEstimator(sklearn.base.BaseEstimator):
    def predict(self, X):
        val = getattr(self, "value", 0.0)
        if hasattr(self, "value_"):
            val = self.value_
        if isinstance(val, (np.ndarray, list)):
            return np.full(X.shape[0], val[0])
        return np.full(X.shape[0], val)

import sksurv.ensemble.survival_loss
sys.modules['sksurv.ensemble.survival_loss'].DummySurvivalEstimator = DummySurvivalEstimator

import os
import gc
import json
import random
import datetime as dt
from pathlib import Path

import pandas as pd
import joblib
from joblib import dump, load

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler as SkScaler
from sklearn.linear_model import LinearRegression

from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from sksurv.ensemble import RandomSurvivalForest, GradientBoostingSurvivalAnalysis
from sksurv.svm import FastKernelSurvivalSVM

from lifelines import WeibullAFTFitter
from lifelines.exceptions import ConvergenceError

import tensorflow as tf

# Optional MTLR (pycox/torch) dependencies
try:
    from pycox.models import MTLR as PycoxMTLR
    from torchtuples.practical import MLPVanilla
    from torchtuples import optim as ttoptim
    from torchtuples.callbacks import EarlyStopping
    import torch
    HAS_PYCOX = True
except Exception:
    HAS_PYCOX = False
    print("[WARNING] PyCox/PyTorch not installed. MTLR predictions will be unavailable.")


### 2. Base Learner Adapters and Helper Functions


In [ ]:
# Define WeibullAFTAdapter to match the saved models class layout
class WeibullAFTAdapter:
    def __init__(self, penalizer=0.1, l1_ratio=0.0):
        self.penalizer = float(penalizer)
        self.l1_ratio = float(l1_ratio)
        self.model = None

    def get_params(self, deep=True):
        return {"penalizer": self.penalizer, "l1_ratio": self.l1_ratio}

    def set_params(self, **params):
        for k, v in params.items():
            setattr(self, k, v)
        return self

    def predict(self, X):
        X = np.asarray(X)
        cols = [f"x{i}" for i in range(X.shape[1])]
        Xdf = pd.DataFrame(X, columns=cols)
        exptime = self.model.predict_expectation(Xdf)
        return -np.asarray(exptime).ravel()

def pred_risk(model, X):
    X = np.asarray(X)
    if hasattr(model, "predict"):
        return np.asarray(model.predict(X)).ravel()
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X)).ravel()
    raise RuntimeError(f"No prediction method for {type(model)}")

def mtlr_risk_scores(model, X_np, cuts):
    surv_df = model.predict_surv_df(X_np)
    if hasattr(np, "trapezoid"):
        return (-np.trapezoid(surv_df.values, x=cuts, axis=0)).astype(np.float64)
    return (-np.trapz(surv_df.values, x=cuts, axis=0)).astype(np.float64)

# sklearn version mismatch compatibility patcher
def patch_sklearn_estimator(model):
    if model is None:
        return
    # Add missing monotonic_cst
    if not hasattr(model, "monotonic_cst"):
        try:
            model.monotonic_cst = None
        except Exception:
            pass
            
    # Add missing link to loss
    if hasattr(model, "_loss") and model._loss is not None:
        if not hasattr(model._loss, "link"):
            from sklearn._loss.link import IdentityLink
            try:
                model._loss.link = IdentityLink()
            except Exception:
                pass
    if hasattr(model, "loss_") and model.loss_ is not None:
        if not hasattr(model.loss_, "link"):
            from sklearn._loss.link import IdentityLink
            try:
                model.loss_.link = IdentityLink()
            except Exception:
                pass

    # Recursively patch base estimators
    if hasattr(model, "estimators_"):
        if isinstance(model.estimators_, np.ndarray):
            for est in model.estimators_.ravel():
                patch_sklearn_estimator(est)
        else:
            for est in model.estimators_:
                patch_sklearn_estimator(est)
                
    if hasattr(model, "init_"):
        patch_sklearn_estimator(model.init_)

# Robust Keras / HDF5 Model Loader
def load_keras_model_robust(model_path):
    model_path = Path(model_path)
    try:
        # Try loading directly
        return tf.keras.models.load_model(model_path.as_posix(), compile=False, safe_mode=False)
    except Exception as e:
        print(f"[INFO] Direct load failed: {e}. Reconstructing model and loading weights...")
        
    h5_path = model_path
    temp_h5_created = False
    if model_path.suffix == ".keras":
        h5_path = model_path.with_suffix(".h5")
        if not h5_path.exists():
            import shutil
            shutil.copy(model_path, h5_path)
            temp_h5_created = True
            
    try:
        import h5py
        with h5py.File(h5_path, "r") as f:
            weight_keys = list(f["model_weights"].keys())
            
        dense_keys = [k for k in weight_keys if "dense" in k]
        dense_keys.sort(key=lambda x: int(x.split("_")[1]))
        
        inputs = tf.keras.Input(shape=(5,), name="input_52")
        x = tf.keras.layers.Dense(128, activation="relu", name=dense_keys[0])(inputs)
        x = tf.keras.layers.Dropout(0.35)(x)
        x = tf.keras.layers.Dense(64, activation="relu", name=dense_keys[1])(x)
        x = tf.keras.layers.Dropout(0.35)(x)
        x = tf.keras.layers.Dense(128, activation="relu", name=dense_keys[2])(x)
        x = tf.keras.layers.Dropout(0.35)(x)
        outputs = tf.keras.layers.Dense(1, activation="linear", name=dense_keys[3])(x)
        
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        model.load_weights(h5_path.as_posix())
        return model
    finally:
        if temp_h5_created and h5_path.exists():
            try:
                os.remove(h5_path)
            except Exception:
                pass

# Parquet Projected Loader
def load_parquet_subset(parquet_path, cpg_list, id_cols=["Share_ID", "Patient_ID", "Subject_ID", "age", "time", "event"]):
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(
        parquet_path,
        thrift_string_size_limit=2_000_000_000,
        thrift_container_size_limit=2_000_000_000
    )
    cols_to_load = [c for c in cpg_list if c in pf.schema.names]
    for col in id_cols:
        if col in pf.schema.names and col not in cols_to_load:
            cols_to_load.append(col)
    table = pf.read(columns=cols_to_load)
    return table.to_pandas()


### 3. Stacked Ensemble Inference Class (Portable)


In [ ]:
class StackedEnsembleInference:
    def __init__(self, models_dir=Path("./pretrained_models"), cpg_csv_path=Path("./coxnet_selected_cpgs_190.csv")):
        self.models_dir = Path(models_dir)
        
        # 1. Load CpG selected feature list
        print("[INFO] Loading CpG selected feature list...")
        cpg_df = pd.read_csv(cpg_csv_path)
        self.cpg_list = cpg_df["CpG"].astype(str).str.strip().dropna().unique().tolist()
        print(f"  Loaded {len(self.cpg_list)} features.")
        
        # 2. Load CpG scaler
        scaler_path = self.models_dir / "cpg_scaler.joblib"
        print(f"[INFO] Loading CpG scaler: {scaler_path}")
        self.cpg_scaler = joblib.load(scaler_path)
        
        # 3. Load base models
        base_dir = self.models_dir / "base_models"
        print(f"[INFO] Loading base models from: {base_dir}")
        self.base_models = {}
        for name in ["rsf", "gbm", "svm", "weibull"]:
            path = base_dir / f"{name}_full.joblib"
            self.base_models[name] = joblib.load(path)
            patch_sklearn_estimator(self.base_models[name])
            print(f"  Loaded base model: {name}")
            
        self.mtlr_model = None
        self.mtlr_cuts = None
        if HAS_PYCOX:
            mtlr_path = base_dir / "mtlr_full.pt"
            if mtlr_path.exists():
                checkpoint = torch.load(mtlr_path, map_location="cpu", weights_only=False)
                cfg = checkpoint["cfg"]
                self.mtlr_cuts = checkpoint["cuts"]
                net = MLPVanilla(len(self.cpg_list), list(cfg["layers"]), cfg["num_durations"], cfg["batchnorm"], cfg["dropout"])
                self.mtlr_model = PycoxMTLR(net, ttoptim.Adam(lr=cfg["lr"]), duration_index=self.mtlr_cuts)
                self.mtlr_model.net.load_state_dict(checkpoint["state_dict"])
                print(f"  Loaded MTLR bundle: {mtlr_path}")
                
        # 4. Load prediction scalers
        scalers_dir = self.models_dir / "pred_scalers"
        print(f"[INFO] Loading prediction scalers...")
        self.pred_scalers = {}
        models_to_scale = ["rsf", "gbm", "svm", "weibull"]
        if self.mtlr_model is not None:
            models_to_scale.append("mtlr")
            
        for name in models_to_scale:
            path = scalers_dir / f"pred_scaler_{name}.joblib"
            self.pred_scalers[name] = joblib.load(path)
            
        # 5. Load stacking meta-learner
        stack_path = self.models_dir / "stacking_meta_learner.keras"
        print(f"[INFO] Loading stacking meta-learner from: {stack_path}")
        self.stacking_model = load_keras_model_robust(stack_path)
        
        # 6. Load age regressor
        reg_path = self.models_dir / "age_regressor.joblib"
        print(f"[INFO] Loading age regressor...")
        self.age_regressor = joblib.load(reg_path)
        
    def predict(self, df_meth):
        """
        Predicts survival risk and calibrated epigenetic age.
        df_meth: pandas DataFrame containing CpG beta value columns.
        """
        df_clean = df_meth.copy()
        missing_cpgs = [c for c in self.cpg_list if c not in df_clean.columns]
        if missing_cpgs:
            print(f"[WARNING] Input DataFrame is missing {len(missing_cpgs)} CpG probes. Imputing with default value (0.5)...")
            for c in missing_cpgs:
                df_clean[c] = 0.5
                
        X_raw = df_clean[self.cpg_list].values
        X_scaled = self.cpg_scaler.transform(X_raw)
        
        raw_preds = {}
        raw_preds["rsf"] = pred_risk(self.base_models["rsf"], X_scaled)
        raw_preds["gbm"] = pred_risk(self.base_models["gbm"], X_scaled)
        raw_preds["svm"] = pred_risk(self.base_models["svm"], X_scaled)
        raw_preds["weibull"] = pred_risk(self.base_models["weibull"], X_scaled)
        
        if self.mtlr_model is not None:
            raw_preds["mtlr"] = mtlr_risk_scores(self.mtlr_model, X_scaled.astype(np.float32), self.mtlr_cuts)
            
        scaled_preds = {}
        for name, pred in raw_preds.items():
            scaled_preds[name] = self.pred_scalers[name].transform(pred.reshape(-1, 1)).ravel()
            
        meta_list = [scaled_preds["rsf"], scaled_preds["gbm"], scaled_preds["svm"], scaled_preds["weibull"]]
        if "mtlr" in scaled_preds:
            meta_list.append(scaled_preds["mtlr"])
            
        X_meta = np.column_stack(meta_list)
        ensemble_risk = self.stacking_model.predict(X_meta.astype(np.float32), verbose=0).ravel()
        age_pred = self.age_regressor.predict(ensemble_risk.reshape(-1, 1)).ravel()
        
        out_df = pd.DataFrame()
        for col in ["Share_ID", "Patient_ID", "Subject_ID"]:
            if col in df_meth.columns:
                out_df[col] = df_meth[col]
                
        if "age" in df_meth.columns:
            out_df["age_true"] = df_meth["age"]
            out_df["age_acceleration"] = age_pred - df_meth["age"]
            
        out_df["ensemble_risk"] = ensemble_risk
        out_df["calibrated_age_pred"] = age_pred
        
        for name in raw_preds.keys():
            out_df[f"pred_{name}"] = raw_preds[name]
            out_df[f"pred_{name}_scaled"] = scaled_preds[name]
            
        return out_df


### 4. Initialize and Run Inference


In [ ]:
# Initialize the pipeline with relative paths
pipeline = StackedEnsembleInference(
    models_dir=Path("./pretrained_models"),
    cpg_csv_path=Path("./coxnet_selected_cpgs_190.csv")
)

# Example: predict on a custom DataFrame:
# predictions = pipeline.predict(my_methylation_df)
print("\nPipeline is initialized and ready to predict!")
